# 데이터 셋 불러오기

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("msambare/fer2013")

print("Path to dataset files:", path)

100%|████████████████████████████████████████████████████████████████████████████████████████████████████| 60.3M/60.3M [00:50<00:00, 1.25MB/s]

Extracting files...


Path to dataset files: C:\Users\msi\.cache\kagglehub\datasets\msambare\fer2013\versions\1


In [1]:
import tensorflow as tf

train_path = r"F:\JINRO_IS_BACK_PROJ\JINRO_PROJ\ai_server\data\train"
test_path = r"F:\JINRO_IS_BACK_PROJ\JINRO_PROJ\ai_server\data\test"

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    train_path,
    image_size=(48,48),
    color_mode="grayscale",
    batch_size=64
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    test_path,
    image_size=(48,48),
    color_mode="grayscale",
    batch_size=64
)

Found 28709 files belonging to 7 classes.
Found 7178 files belonging to 7 classes.


In [5]:
len(train_ds)

449

In [6]:
len(val_ds)

113

# 데이터 정규화

In [7]:
normalization_layer = tf.keras.layers.Rescaling(1./255)

train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y))

# CNN 모델 생성

In [8]:
from tensorflow.keras import layers, models

model = models.Sequential([

    layers.Conv2D(32,(3,3),activation="relu",input_shape=(48,48,1)),
    layers.MaxPooling2D(),

    layers.Conv2D(64,(3,3),activation="relu"),
    layers.MaxPooling2D(),

    layers.Conv2D(128,(3,3),activation="relu"),
    layers.MaxPooling2D(),

    layers.Flatten(),

    layers.Dense(128,activation="relu"),
    layers.Dropout(0.5),

    layers.Dense(7,activation="softmax")

])

F:\PJ\.venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [9]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [10]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30
)

Epoch 1/30
449/449 ━━━━━━━━━━━━━━━━━━━━ 40s 85ms/step - accuracy: 0.2935 - loss: 1.7436 - val_accuracy: 0.4004 - val_loss: 1.5613
Epoch 2/30
449/449 ━━━━━━━━━━━━━━━━━━━━ 41s 85ms/step - accuracy: 0.4085 - loss: 1.5304 - val_accuracy: 0.4613 - val_loss: 1.4066
Epoch 3/30
449/449 ━━━━━━━━━━━━━━━━━━━━ 22s 41ms/step - accuracy: 0.4514 - loss: 1.4221 - val_accuracy: 0.4840 - val_loss: 1.3491
Epoch 4/30
449/449 ━━━━━━━━━━━━━━━━━━━━ 10s 22ms/step - accuracy: 0.4869 - loss: 1.3428 - val_accuracy: 0.5081 - val_loss: 1.2920
Epoch 5/30
449/449 ━━━━━━━━━━━━━━━━━━━━ 12s 26ms/step - accuracy: 0.5101 - loss: 1.2858 - val_accuracy: 0.5210 - val_loss: 1.2407
Epoch 6/30
449/449 ━━━━━━━━━━━━━━━━━━━━ 43s 75ms/step - accuracy: 0.5327 - loss: 1.2298 - val_accuracy: 0.5348 - val_loss: 1.2066
Epoch 7/30
449/449 ━━━━━━━━━━━━━━━━━━━━ 44s 82ms/step - accuracy: 0.5539 - loss: 1.1889 - val_accuracy: 0.5364 - val_loss: 1.1978
Epoch 8/30
449/449 ━━━━━━━━━━━━━━━━━━━━ 42s 83ms/step - accuracy: 0.5681 - loss: 1.1485 - 

In [12]:
loss, acc = model.evaluate(train_ds)
print("test accuracy:", acc)

449/449 ━━━━━━━━━━━━━━━━━━━━ 16s 35ms/step - accuracy: 0.8351 - loss: 0.4660
test accuracy: 0.8350691199302673


In [13]:
loss, acc = model.evaluate(val_ds)
print("test accuracy:", acc)

113/113 ━━━━━━━━━━━━━━━━━━━━ 4s 35ms/step - accuracy: 0.5581 - loss: 1.5992
test accuracy: 0.5580942034721375


In [ ]:
model.save(r"F:\JINRO_IS_BACK_PROJ\JINRO_PROJ\ai_server\app\models\emotion_cnn.h5")

# 정확도 높이기

In [15]:
from tensorflow.keras.layers import RandomFlip, RandomRotation, RandomZoom

data_augmentation = tf.keras.Sequential([
    RandomFlip("horizontal"),
    RandomRotation(0.1),
    RandomZoom(0.1)
])

train_ds = train_ds.map(lambda x,y:(data_augmentation(x),y))

In [16]:
optimizer = tf.keras.optimizers.Adam(
    learning_rate=0.0003
)

In [17]:
callback = tf.keras.callbacks.EarlyStopping(
    patience=5,
    restore_best_weights=True
)

In [ ]:
model = tf.keras.Sequential([

    layers.Conv2D(32,(3,3),activation="relu",input_shape=(48,48,1)),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.Conv2D(64,(3,3),activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.Conv2D(128,(3,3),activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.Conv2D(256,(3,3),activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(),

    layers.Flatten(),

    layers.Dense(256,activation="relu"),
    layers.Dropout(0.5),

    layers.Dense(7,activation="softmax")
])

In [ ]:
model.compile(
    optimizer=optimizer,
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=40,
    callbacks=[callback]
)